# Transformers


#### A Transformer is a deep learning architecture designed to process sequences of text using self-attention.

- Transformers are the foundation of many modern NLP models, including BERT, GPT, and T5.

- Before Transformers, we used models such as RNNs and LSTMs for NLP tasks.

<img src="https://drive.google.com/thumbnail?id=1unPqGaKftHxp37-F99Bg6hJzFbQdCGC7&sz=w1000" width="700">

In [ ]:
%pip install -q "transformers>=4.45,<5" torch matplotlib

## 1. Load a Transformer

We use **DistilBERT**, a  Transformer fine-tuned for positive/negative sentiment classification.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

torch.manual_seed(42)
torch.set_num_threads(2)
model_name = "distilbert-base-uncased-finetuned-sst-2-english"
# The first run requires access to Hugging Face to download the model files.
tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir="model_cache")
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, cache_dir="model_cache", attn_implementation="eager", use_safetensors=True
)
model.eval()
print("Transformer layers:", model.config.n_layers)
print("Attention heads per layer:", model.config.n_heads)
print("Hidden size:", model.config.dim)


<img src="https://drive.google.com/thumbnail?id=1v2dRK1KR_ydZOknjj8Xv8_wjOdXleDYs&sz=w1000" width="500">

## 2. Tokenizing the text

In [ ]:
sentence = "The movie was not good."

# Tokenization: converts text into token IDs
inputs = tokenizer(sentence, return_tensors="pt")

# [CLS] = start token, [SEP] = end token
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

print("Tokens:", tokens)

## 3. Embeddings and self-attention

Each token ID becomes a **vector (embedding)**. Inside each Transformer layer, **self-attention** lets a token use information from other tokens in the same sentence.

In [ ]:
# Eager attention was selected while loading the model so weights are available.
with torch.no_grad():
    embeddings = model.distilbert.embeddings(inputs["input_ids"])
    outputs = model(**inputs, output_attentions=True)

attention = outputs.attentions[0][0, 0]
print("Embeddings shape:", embeddings.shape)
print("Attention shape:", attention.shape)
print("Tokens:", tokens)
print("Attention weights:\n", attention)

## 4. Classification: From Transformer to prediction

The Transformer produces contextual representations. The **classification head** turns them into two scores (*logits*). **Softmax** converts these into probabilities that sum to 1.

In [ ]:
# Get prediction scores from the Transformer
with torch.no_grad():
    logits = model(**inputs).logits

# Convert scores into probabilities
probs = torch.softmax(logits, dim=1)[0]

# Display the prediction
prediction = probs.argmax().item()

print("Prediction:", model.config.id2label[prediction])
print("Confidence:", round(probs[prediction].item(), 3))



### Task: Exploring Transformer Layers and Attention Heads

Load the pretrained **`bert-base-uncased`** model using Hugging Face Transformers.

1. Write code to print the number of Transformer layers and the number of attention heads per layer.
2. Calculate and print the total number of attention heads across all Transformer layers.
3. Using the sentence `"Transformers use self-attention to understand language."`, extract the attention weights from the **third Transformer layer and second attention head**.
4. Explain what the rows and columns of the attention matrix represent.

### 1–3. Inspect BERT Layers and Extract the Requested Attention
Load `bert-base-uncased` separately from the DistilBERT lesson model. Select eager attention to expose its attention weights. The model uses zero-based indexing, so the third layer is index `2` and the second head is index `1`. The middle index selects the first sentence in the batch.

In [ ]:
from transformers import AutoModel, AutoTokenizer
import torch
import matplotlib.pyplot as plt

bert_model_name = "bert-base-uncased"
bert_tokenizer = AutoTokenizer.from_pretrained(bert_model_name, cache_dir="model_cache")
bert_model = AutoModel.from_pretrained(
    bert_model_name, cache_dir="model_cache", attn_implementation="eager", use_safetensors=True
)
bert_model.eval()

In [ ]:
def inspect_bert_attention(bert_model, bert_tokenizer, sentence):
    number_of_layers = bert_model.config.num_hidden_layers
    heads_per_layer = bert_model.config.num_attention_heads
    if number_of_layers < 3 or heads_per_layer < 2:
        raise ValueError("The requested third layer and second head are not present.")

    encoded = bert_tokenizer(sentence, return_tensors="pt")
    bert_model.eval()
    with torch.no_grad():
        result = bert_model(**encoded, output_attentions=True)
    if result.attentions is None:
        raise RuntimeError("Load the model with attn_implementation='eager'.")

    # [layer][batch, head, query token, key token]
    selected_attention = result.attentions[2][0, 1].detach().cpu()
    token_labels = bert_tokenizer.convert_ids_to_tokens(encoded["input_ids"][0].tolist())
    details = {
        "layers": number_of_layers,
        "heads_per_layer": heads_per_layer,
        "total_heads": number_of_layers * heads_per_layer,
    }
    return details, token_labels, selected_attention

In [ ]:
sentence = "Transformers use self-attention to understand language."
bert_details, bert_tokens, bert_attention = inspect_bert_attention(
    bert_model, bert_tokenizer, sentence
)

print("Transformer layers:", bert_details["layers"])
print("Attention heads per layer:", bert_details["heads_per_layer"])
print("Total attention heads across all layers:", bert_details["total_heads"])
print("Selected: third Transformer layer, second attention head")
print("Tokens:", bert_tokens)
print("Attention matrix shape:", tuple(bert_attention.shape))
print("Attention matrix:")
print(bert_attention)
print("Row sums:", bert_attention.sum(dim=-1))

BERT-base has 12 Transformer layers and 12 heads per layer, giving **12 × 12 = 144 attention-head instances across its layers**. These heads operate in parallel within each layer; this does not mean that one layer contains 144 heads. The code reads these values from the loaded configuration rather than hardcoding the result.

### Visualize the Selected Attention Matrix
The heatmap shows the actual matrix returned for the requested layer and head. Use the tokenizer's labels, including special tokens and any subword pieces, so each axis matches the matrix dimensions.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
heatmap = ax.imshow(bert_attention.numpy(), cmap="viridis", vmin=0, vmax=1)
positions = range(len(bert_tokens))
ax.set_xticks(positions, bert_tokens, rotation=60, ha="right")
ax.set_yticks(positions, bert_tokens)
ax.set_xlabel("Key tokens — tokens being attended to")
ax.set_ylabel("Query tokens — tokens attending")
ax.set_title("BERT: third Transformer layer, second attention head")
fig.colorbar(heatmap, ax=ax, label="Attention weight")
fig.tight_layout()
plt.show()

### 4. What Do the Rows and Columns Represent?
- Each **row** represents a query token: the token whose representation is being updated.
- Each **column** represents a key token: a token that the query can attend to.
- Entry **[i, j]** is the attention weight from query token **i** to key token **j**. It determines the contribution of that position's value vector to the head's output.
- A larger weight means a stronger contribution within that query row. With the model in evaluation mode and this unpadded sentence, each row sums to approximately **1**. Columns need not sum to 1, and the matrix need not be symmetric.
- The axes include `[CLS]`, `[SEP]`, punctuation, and any WordPiece subwords; matrix dimensions count tokens, not just whole words.

This is one head in one layer. It illustrates a learned attention pattern, rather than proving which words caused a prediction. The BERT model used for this task is a base encoder, while the earlier DistilBERT lesson model has a sentiment-classification head.